# INA700 - train the shape detector

Author: Suraj Karki  
Course: INA700 - AI-based pick and place

Trains YOLOv8n to find **circle, square and star** on the A3 sheet, and exports
it for the Raspberry Pi.

Hexagon and triangle are deliberately **not** classes. They appear in the photos
unlabelled, so the model learns them as background and refuses to report them.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`.

**Input:** `yolo.zip`, produced on your laptop by
```
cd pi
python tools/split_dataset.py
```

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install Ultralytics

Pinned, so a future release cannot change the result you write up.

In [ ]:
%pip install -q ultralytics==8.3.40

import ultralytics
ultralytics.checks()

## 3. Upload the dataset

Zip `model/dataset/yolo` on your laptop first, then upload it here.

Colab discards everything when the session ends, so keep the zip on your laptop.

In [ ]:
from google.colab import files
import zipfile, pathlib

uploaded = files.upload()          # choose yolo.zip
name = next(iter(uploaded))

with zipfile.ZipFile(name) as archive:
    archive.extractall('/content/dataset')

for path in sorted(pathlib.Path('/content/dataset').rglob('data.yaml')):
    print(path)

## 4. Point `data.yaml` at Colab

`split_dataset.py` writes the path it ran under, which is on your laptop.
Ultralytics resolves `train:` and `val:` relative to `path:`, so it has to be
rewritten here or training silently finds no images.

In [ ]:
import yaml, pathlib

root = next(pathlib.Path('/content/dataset').rglob('data.yaml')).parent
config = yaml.safe_load((root / 'data.yaml').read_text())
config['path'] = str(root)
(root / 'data.yaml').write_text(yaml.safe_dump(config, sort_keys=False))

DATA = str(root / 'data.yaml')
print(open(DATA).read())

## 5. Look at the data before training it

Counts first, then one image with its boxes drawn on. If a box is in the wrong
place here, no amount of training fixes it.

In [ ]:
import pathlib, collections

names = config['names']

for split in ('train', 'val'):
    labels = sorted((root / split / 'labels').glob('*.txt'))
    per_class = collections.Counter()
    empty = 0
    for path in labels:
        rows = [r for r in path.read_text().splitlines() if r.strip()]
        if not rows:
            empty += 1
        for row in rows:
            per_class[names[int(row.split()[0])]] += 1
    print(f'{split:5s} {len(labels):4d} images, {empty:3d} empty   {dict(per_class)}')

In [ ]:
import cv2, random, matplotlib.pyplot as plt

def show(split='train', count=3):
    images = sorted(p for p in (root / split / 'images').iterdir()
                    if p.suffix.lower() in ('.jpg', '.png'))
    fig, axes = plt.subplots(1, count, figsize=(6 * count, 5))
    for axis, path in zip(axes, random.sample(images, count)):
        image = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        height, width = image.shape[:2]
        label = root / split / 'labels' / f'{path.stem}.txt'
        for row in label.read_text().splitlines():
            if not row.strip():
                continue
            class_id, cx, cy, w, h = row.split()
            cx, cy, w, h = float(cx) * width, float(cy) * height, float(w) * width, float(h) * height
            p0 = (int(cx - w / 2), int(cy - h / 2))
            p1 = (int(cx + w / 2), int(cy + h / 2))
            cv2.rectangle(image, p0, p1, (255, 0, 0), 6)
            cv2.putText(image, names[int(class_id)], (p0[0], p0[1] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 2.0, (255, 0, 0), 4)
        axis.imshow(image); axis.set_title(path.name); axis.axis('off')
    plt.tight_layout(); plt.show()

show('train')

## 6. Train

**Why these augmentation settings.**

The model only has to *find* each object and say which of the three it is.
Orientation is measured separately, by the harmonic estimator working on the
outline. That frees the training to rotate and flip as much as it likes -
a system that regressed the angle from the network could not do this.

| Setting | Value | Reason |
|---|---|---|
| `degrees` | 180 | objects land at any angle on the sheet |
| `fliplr`, `flipud` | 0.5 | free variety; orientation is not the model's job |
| `hsv_v` | 0.5 | the sheet ranged from 70 to 209 across sessions - cover it |
| `hsv_h` | 0.015 | colours must stay recognisable; the HSV classifier depends on them |
| `scale` | 0.3 | the camera height is fixed, so scale barely varies in reality |
| `mosaic` | 1.0 | teaches several objects per frame, which the run-time board has |
| `imgsz` | 640 | objects are about 250 px in a 2448 px frame, so 65 px here |
| `patience` | 25 | stops when val stops improving, rather than over-fitting 280 images |

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')      # nano: the Pi has to run this

results = model.train(
    data=DATA,
    epochs=150,
    patience=25,
    imgsz=640,
    batch=16,
    seed=1,
    project='/content/runs',
    name='ina700',
    degrees=180.0,
    fliplr=0.5,
    flipud=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.5,
    scale=0.3,
    translate=0.1,
    mosaic=1.0,
    plots=True,
)

## 7. Read the result

What to look for:

- **mAP50 above 0.95** on three large, well-separated shapes. Below 0.9, something is wrong with the labels, not the training.
- **The confusion matrix background column.** Hexagons and triangles landing in a shape column means the hard negatives did not take - shoot more mixed scenes.
- **Per-class recall for `square`.** It has the fewest photos, so it fails first.

In [ ]:
metrics = model.val()
print(f'mAP50    {metrics.box.map50:.3f}')
print(f'mAP50-95 {metrics.box.map:.3f}')
for index, name in names.items():
    print(f'  {name:8s} P {metrics.box.p[index]:.3f}  R {metrics.box.r[index]:.3f}')

In [ ]:
from IPython.display import Image, display
import pathlib

run = pathlib.Path('/content/runs/ina700')
for plot in ('confusion_matrix_normalized.png', 'results.png', 'val_batch0_pred.jpg'):
    path = run / plot
    if path.exists():
        print(plot)
        display(Image(str(path), width=900))

## 8. Export for the Pi

`best.pt` runs anywhere. NCNN is the format that makes a Pi 4 usable - roughly
two to three times faster than the PyTorch weights on ARM CPU.

In [ ]:
best = '/content/runs/ina700/weights/best.pt'

YOLO(best).export(format='ncnn', imgsz=640)

!cd /content/runs/ina700/weights && zip -qr /content/ina700_model.zip best.pt best_ncnn_model
!ls -la /content/ina700_model.zip

In [ ]:
from google.colab import files
files.download('/content/ina700_model.zip')

## 9. Put it on the laptop

Unzip into `model/`, then in `pi/config/config.yaml`:

```yaml
detector:
  kind: yolo
  model_path: ../model/best.pt          # or ../model/best_ncnn_model on the Pi
  image_size: 640                       # must match the imgsz trained above
  confidence: 0.5
```

Then, with the camera running:

```
cd pi
python tools/preflight.py
python smoke_test.py
```